In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer


import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.activations import relu,linear
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam


In [2]:
#Training Data.
#Skip the header as it is not a number.
#Each row corresponds to a movie and displays its runtime, IMDB rating, IMDB number of votes,...
X = np.genfromtxt('Training_X.csv', delimiter=',', skip_header=1)  
m, n = X.shape
#Each row of y says if 2.5*movie_budget was greater than worldwide_gross. This is the metric we are using to 
#determine if a movie was successful.
y = np.genfromtxt('Training_Y.csv', delimiter=',', skip_header=1)

In [3]:
#Splitting data to have a part of it for testing and cross-validation
#I am saving 30% of the total data for that purpose.
X_train, X_, y_train, y_ = train_test_split(X,y,test_size=0.3, random_state=1)

#Then split into test and cross-validation that 30%. Half goes to cv half to test.
X_cv, X_test, y_cv, y_test = train_test_split(X_,y_,test_size=0.50, random_state=1)

#Normalize the data so that features are on the same range.
#scaler = MinMaxScaler()
#scaler.fit(X_train)
#X_train_norm=scaler.transform(X_train)

#Transform test and cross-validation data as well
#X_cv_norm=scaler.transform(X_cv)
#X_test_norm=scaler.transform(X_test)

In [4]:
#Normalize the data so that features are on the same range.
#Tried also standard scaler.
scaler= StandardScaler()
scaler.fit(X_train)
X_train_norm=scaler.transform(X_train)

#Transform test and cross-validation data as well
X_cv_norm=scaler.transform(X_cv)
X_test_norm=scaler.transform(X_test)


In [6]:
#Multilayer perceptron to be trained on the data.
#Input size is the number of features of each example.
#, kernel_regularizer=regularizers.l2(0.01)
model = Sequential(                      
    [                                   
        tf.keras.Input(shape=(n,)),  
        Dense(40, activation='relu'),
        Dense(1,  activation='linear')
    ], name = "my_model"                                    
) 
#I made the last layer linear and apply the sigmoid manually.

#Define a loss function and use adaptative alpha (Adam) to perform gradient descent. 
#It is important to specify from_logits=True as the last layer is not a sigmoid. To make it numerically stable
#the sigmoid is in that way applied when computing the cost. 
model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(0.001),
)

#I use early_stopping to allow a fast evaluation of any one model.
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',      
    patience=5,              # epochs to wait before stopping
    restore_best_weights=True
)

history = model.fit(
    X_train_norm, y_train,
    validation_data=(X_cv_norm, y_cv),
    epochs=1000,
    callbacks=[early_stopping]
)
'''
model.fit(
    X_train_norm, y_train,
    validation_data=(X_cv_norm, y_cv),
    epochs=170
)
'''

2026-05-19 13:58:04.667904: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-19 13:58:04.683589: I tensorflow/core/common_runtime/process_util.cc:146] Creating new thread pool with default inter op setting: 2. Tune using inter_op_parallelism_threads for best performance.


Epoch 1/1000
138/138 [==============================] - 6s 36ms/step - loss: 0.6871 - val_loss: 0.6191
Epoch 2/1000
138/138 [==============================] - 6s 42ms/step - loss: 0.6091 - val_loss: 0.6017
Epoch 3/1000
138/138 [==============================] - 5s 40ms/step - loss: 0.5937 - val_loss: 0.5964
Epoch 4/1000
138/138 [==============================] - 5s 39ms/step - loss: 0.5852 - val_loss: 0.5920
Epoch 5/1000
138/138 [==============================] - 5s 39ms/step - loss: 0.5799 - val_loss: 0.5887
Epoch 6/1000
138/138 [==============================] - 5s 39ms/step - loss: 0.5758 - val_loss: 0.5884
Epoch 7/1000
138/138 [==============================] - 6s 40ms/step - loss: 0.5710 - val_loss: 0.5870
Epoch 8/1000
138/138 [==============================] - 5s 40ms/step - loss: 0.5680 - val_loss: 0.5852
Epoch 9/1000
138/138 [==============================] - 6s 41ms/step - loss: 0.5645 - val_loss: 0.5839
Epoch 10/1000
138/138 [==============================] - 5s 37ms/step - l

'\nmodel.fit(\n    X_train_norm, y_train,\n    validation_data=(X_cv_norm, y_cv),\n    epochs=170\n)\n'

In [7]:
val_loss = model.evaluate(X_cv_norm, y_cv)
print("Validation loss:", val_loss)

30/30 [==============================] - 1s 21ms/step - loss: 0.5813
Validation loss: 0.581280529499054


In [8]:
#Routine to make predictions for X_input
def model_predict(X_input):
    '''
     X_input    : (ndarray )  Array of features to predict
     returns yhat a numpy array of 0/1 values corresponding to predictions.
    '''    
    #Making predictions.
    prediction = model.predict(X_input)
    #Applying the sigmoid at the end manually.
    probs = tf.nn.sigmoid(prediction).numpy()
    
    #Convert to 0/1 predictions
    yhat=np.zeros(len(probs))
    for i in range(len(probs)):    
        #Threshold to convert the predictions into TRUE/FALSE outputs.
        if probs[i] >= 0.5:
            yhat[i] = 1
        else:
            yhat[i] = 0
    return yhat
    

In [9]:
# Classification Error.
def eval_cat_err(y, yhat):
    """ 
      y    : (ndarray  Shape (m,) or (m,1))  target value of each example
      yhat : (ndarray  Shape (m,) or (m,1))  predicted value of each example           
    """
    m = len(y)
    incorrect = 0
    for i in range(m):
     
        if(y[i]!=yhat[i]):
            incorrect+=1
            
    cerr=incorrect/m
     
    
    return(cerr)

#Error on training set and cross-validation set.
training_cerr = eval_cat_err(y_train, model_predict(X_train_norm))
cv_cerr = eval_cat_err(y_cv, model_predict(X_cv_norm))


print("Training Error")
print(training_cerr)
print("Cross-validation Error")
print(cv_cerr)
print("With weights")
print(model.get_weights())

30/30 [==============================] - 1s 28ms/step
Training Error
0.27311974551238355
Cross-validation Error
0.2926829268292683
With weights
[array([[-0.14784987,  0.06002428,  0.14450043, ..., -0.12812527,
         0.13400425,  0.12477536],
       [-0.07987748,  0.5246847 ,  0.08415023, ...,  0.1663857 ,
         0.10184146, -0.12504943],
       [-0.12737486,  0.06780804,  0.14823984, ...,  0.2635747 ,
        -0.35606003,  0.15277193],
       ...,
       [ 0.07366548, -0.19982104, -0.04882922, ..., -0.26932612,
         0.21509182,  0.074774  ],
       [-0.04611715,  0.24445282,  0.16895103, ..., -0.62548417,
        -0.22614832,  0.1570752 ],
       [ 0.06518118,  0.10046911,  0.13186587, ...,  0.02191679,
        -0.10342669, -0.28192323]], dtype=float32), array([ 0.1787665 , -0.04810326, -0.03424579, -0.03162245,  0.09367602,
       -0.04562943,  0.10213485, -0.19465306,  0.09314543, -0.10679641,
       -0.10070712,  0.04949838, -0.03690723,  0.16586712, -0.08717503,
        0.

In [10]:
#Error for reporting model accuracy
accuracy = eval_cat_err(y_test, model_predict(X_test_norm))
print(accuracy)

30/30 [==============================] - 1s 21ms/step
0.3072033898305085


In [11]:
#Now we test data from 2023
#Training Data.
#Skip the header as it is not a number.
#Each row corresponds to a movie and displays its runtime, IMDB rating, IMDB number of votes,...
X_1 = np.genfromtxt('Hypothesis_X.csv', delimiter=',', skip_header=1)  
m_1, n_1 = X_1.shape
#Each row of y says if 2.5*movie_budget was greater than worldwide_gross. This is the metric we are using to 
#determine if a movie was successful.
y_1 = np.genfromtxt('Hypothesis_Y.csv', delimiter=',', skip_header=1)

#We have to scale before predicting
X_1_test=scaler.transform(X_1)

data2023_cerr = eval_cat_err(y_1, model_predict(X_1_test))


print("2023 Error")
print(data2023_cerr)

5/5 [==============================] - 0s 25ms/step
2023 Error
0.2945205479452055
